<a href="https://colab.research.google.com/github/aravg-df/aravg-df/blob/main/Fine_Tune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Enable T4 GPU in Colab."
    )

gpu_name = torch.cuda.get_device_name(0)

gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("GPU:", gpu_name)
print("GPU memory:", round(gpu_memory, 2), "GB")


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
!pip install -U transformers

In [ ]:

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
import torch

MODEL_NAME = "Qwen/Qwen3-1.7B"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Download and load pretrained Qwen
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.float16
)

print("Qwen3-1.7B loaded successfully!")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3-1.7B loaded successfully!


In [ ]:

total_params = sum(
    p.numel() for p in model.parameters()
)

print("Model:", MODEL_NAME)
print(f"Total parameters: {total_params:,}")
print(f"Parameters (billions): {total_params/1e9:.2f}B")
print("Model device:", model.device)

allocated = torch.cuda.memory_allocated() / 1024**3

print(f"GPU memory allocated: {allocated:.2f} GB")


Model: Qwen/Qwen3-1.7B
Total parameters: 1,015,931,904
Parameters (billions): 1.02B
Model device: cuda:0
GPU memory allocated: 4.47 GB


In [ ]:
def ask_qwen(question, max_new_tokens=128):

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


question = (
    "What is the age of Arjun Verma "
    "in my private dataset?"
)

print("Question:", question)
print("Answer:", ask_qwen(question))

Question: What is the age of Arjun Verma in my private dataset?
Answer: I cannot access or retrieve personal data, including the age of individuals, including Arjun Verma, from a private dataset. I cannot provide information that is not publicly available or that requires access to sensitive or private data.

If you have any questions about public information or need assistance with something else, feel free to ask!


In [ ]:

from google.colab import files
import os

os.makedirs("/content/data", exist_ok=True)

uploaded = files.upload()

for filename, content in uploaded.items():
    destination = os.path.join("/content/data", filename)

    with open(destination, "wb") as f:
        f.write(content)

    print("Uploaded:", destination)


Saving people_100k_records.csv to people_100k_records.csv
Saving train_people_100k.jsonl to train_people_100k.jsonl
Saving val_people_10k.jsonl to val_people_10k.jsonl
Uploaded: /content/data/people_100k_records.csv
Uploaded: /content/data/train_people_100k.jsonl
Uploaded: /content/data/val_people_10k.jsonl


In [ ]:

import pandas as pd

CSV_PATH = "/content/data/people_100k_records.csv"

df = pd.read_csv(
    CSV_PATH,
    dtype=str,
    keep_default_na=False
)

required_columns = [
    "person_id",
    "name",
    "father_name",
    "mother_name",
    "age",
    "aadhaar_number",
    "pan_number"
]

assert all(c in df.columns for c in required_columns)
assert df["person_id"].is_unique

print("Total records:", len(df))
print("Unique names:", df["name"].nunique())
print("Duplicate names:", df["name"].duplicated().sum())
print("Missing values:")
print(df[required_columns].eq("").sum())

display(df.head())


Total records: 100000
Unique names: 3000
Duplicate names: 97000
Missing values:
person_id         0
name              0
father_name       0
mother_name       0
age               0
aadhaar_number    0
pan_number        0
dtype: int64


,person_id,name,father_name,mother_name,age,aadhaar_number,pan_number
0,P00001,Arjun Sharma,Rajesh Sharma,Sunita Sharma,47,997228186620,RBLXA4104K
1,P00002,Kavya Verma,Suresh Verma,Kavita Verma,64,950040793371,TIKXK8596M
2,P00003,Rohan Patel,Mahesh Patel,Poonam Patel,52,908884442057,MUFXR0678Y
3,P00004,Priya Gupta,Ramesh Gupta,Meena Gupta,51,942377826025,RSLXP3524I
4,P00005,Amit Singh,Dinesh Singh,Shanti Singh,47,956286439063,BVBXA1050M


In [ ]:

import json
from collections import defaultdict

TRAIN_PATH = "/content/data/train_people_100k.jsonl"
VAL_PATH = "/content/data/val_people_10k.jsonl"

def read_jsonl(path):
    valid = []
    invalid = []

    with open(path, "r", encoding="utf-8-sig") as f:
        for line_number, line in enumerate(f, 1):
            try:
                record = json.loads(line)
                messages = record["messages"]

                roles = [m["role"] for m in messages]

                if roles != ["system", "user", "assistant"]:
                    raise ValueError("Unexpected message roles")

                if not all(
                    isinstance(m["content"], str)
                    and m["content"].strip()
                    for m in messages
                ):
                    raise ValueError("Empty or invalid content")

                valid.append(record)

            except (ValueError, KeyError, TypeError) as e:
                invalid.append((line_number, str(e)))

    return valid, invalid


train_raw, train_errors = read_jsonl(TRAIN_PATH)
val_raw, val_errors = read_jsonl(VAL_PATH)

print("Valid training examples:", len(train_raw))
print("Valid validation examples:", len(val_raw))
print("Training errors:", train_errors[:10])
print("Validation errors:", val_errors[:10])

# Deduplicate exact question-answer pairs
unique = {}

for record in train_raw + val_raw:
    messages = record["messages"]
    question = messages[1]["content"].strip()
    answer = messages[2]["content"].strip()

    unique[(question, answer)] = record

# Detect questions with contradictory answers
answers_by_question = defaultdict(set)

for question, answer in unique:
    answers_by_question[question].add(answer)

conflicting_questions = {
    q for q, answers in answers_by_question.items()
    if len(answers) > 1
}

clean_examples = [
    record
    for (question, answer), record in unique.items()
    if question not in conflicting_questions
]

print("Unique question-answer pairs:", len(unique))
print("Conflicting questions:", len(conflicting_questions))
print("Clean examples:", len(clean_examples))


Valid training examples: 61312
Valid validation examples: 10000
Training errors: [(61313, 'Unterminated string starting at: line 1 column 45 (char 44)')]
Validation errors: []
Unique question-answer pairs: 61304
Conflicting questions: 7913
Clean examples: 36970


In [ ]:

from sklearn.model_selection import GroupShuffleSplit

questions = [
    record["messages"][1]["content"].strip()
    for record in clean_examples
]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(clean_examples, groups=questions)
)

train_examples = [clean_examples[i] for i in train_idx]
val_examples = [clean_examples[i] for i in val_idx]

train_questions = {
    x["messages"][1]["content"]
    for x in train_examples
}

val_questions = {
    x["messages"][1]["content"]
    for x in val_examples
}

assert train_questions.isdisjoint(val_questions)

print("Training examples:", len(train_examples))
print("Validation examples:", len(val_examples))
print("Overlapping questions:", len(train_questions & val_questions))


Training examples: 33273
Validation examples: 3697
Overlapping questions: 0


In [ ]:

import os
import json

OUTPUT_DIR = "/content/data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(
                json.dumps(record, ensure_ascii=False)
                + "\n"
            )

save_jsonl(
    train_examples,
    f"{OUTPUT_DIR}/train.jsonl"
)

save_jsonl(
    val_examples,
    f"{OUTPUT_DIR}/validation.jsonl"
)

print("Cleaned datasets saved successfully!")


Cleaned datasets saved successfully!


In [ ]:

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": f"{OUTPUT_DIR}/train.jsonl",
        "validation": f"{OUTPUT_DIR}/validation.jsonl"
    }
)

print(dataset)

sample = dataset["train"][0]

for message in sample["messages"]:
    print(
        message["role"].upper(),
        ":",
        message["content"]
    )


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 33273
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 3697
    })
})
SYSTEM : You are an accurate Citizen Information and KYC Assistant.
USER : Generate structured JSON profile for person ID P00003.
ASSISTANT : {
  "person_id": "P00003",
  "name": "Rohan Patel",
  "father_name": "Mahesh Patel",
  "mother_name": "Poonam Patel",
  "age": 52,
  "aadhaar_number": "908884442057",
  "pan_number": "MUFXR0678Y"
}


In [ ]:

from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/Qwen_FineTuning"

os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copytree(
    OUTPUT_DIR,
    os.path.join(DRIVE_DIR, "processed"),
    dirs_exist_ok=True
)

print("Processed datasets backed up to Google Drive.")


Mounted at /content/drive
Processed datasets backed up to Google Drive.


In [ ]:

import json

person_id = "P00003"

csv_record = df[
    df["person_id"] == person_id
]

print("CSV source record:")
print(csv_record.to_string(index=False))

print("\nTraining example:")

for example in dataset["train"]:
    if f"person ID {person_id}" in example["messages"][1]["content"]:
        print(example["messages"][2]["content"])
        break


CSV source record:
person_id        name  father_name  mother_name age aadhaar_number pan_number
   P00003 Rohan Patel Mahesh Patel Poonam Patel  52   908884442057 MUFXR0678Y

Training example:
{
  "person_id": "P00003",
  "name": "Rohan Patel",
  "father_name": "Mahesh Patel",
  "mother_name": "Poonam Patel",
  "age": 52,
  "aadhaar_number": "908884442057",
  "pan_number": "MUFXR0678Y"
}


In [ ]:

import torch
import transformers
import peft
import trl
import datasets

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

assert torch.cuda.is_available(), "Enable T4 GPU in Colab"

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

print("Training examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))


PyTorch: 2.14.1+cu130
Transformers: 5.24.0
PEFT: 0.19.1
TRL: 0.29.1
GPU: Tesla T4
GPU memory: 14.56 GB
Training examples: 33273
Validation examples: 3697


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "/content/data/processed/train.jsonl",
        "validation": "/content/data/processed/validation.jsonl"
    }
)

In [ ]:

import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen3-1.7B"

# Release an earlier model if present
if "model" in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map={"": 0},
    dtype=torch.float16
)

model.config.use_cache = False

print("Model loaded successfully")
print("4-bit quantization:",
      model.is_loaded_in_4bit)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Model loaded successfully
4-bit quantization: True


In [ ]:

from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

print("Quantized model prepared for training.")


Quantized model prepared for training.


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3204: FutureWarning: `gradient_checkpointing_enable` is deprecated and will be removed in v6. Use `activation_checkpointing_enable` instead.
  warnings.warn(


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

print("LoRA adapters attached successfully!")
model.print_trainable_parameters()

LoRA adapters attached successfully!
trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3312: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [ ]:

import numpy as np

sample_size = min(1000, len(dataset["train"]))

sample = dataset["train"].shuffle(seed=42).select(
    range(sample_size)
)

lengths = []

for example in sample:
    tokens = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
        enable_thinking=False
    )

    lengths.append(len(tokens))

print("Average tokens:", round(np.mean(lengths), 1))
print("95th percentile:", np.percentile(lengths, 95))
print("Maximum sampled length:", max(lengths))


Average tokens: 2.0
95th percentile: 2.0
Maximum sampled length: 2


In [36]:

from trl import SFTConfig
import inspect

OUTPUT_DIR = "/content/qwen3_personal_lora"

# Check supported parameters for installed TRL version
supported = inspect.signature(SFTConfig).parameters

config = {
    "output_dir": OUTPUT_DIR,
    "max_steps": 100,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "optim": "paged_adamw_8bit",
    "lr_scheduler_type": "cosine",
    "warmup_steps": 3,
    "weight_decay": 0.01,
    "max_grad_norm": 0.3,
    "fp16": True,
    "bf16": False,
    "max_length": 512,
    "packing": False,
    "assistant_only_loss": True,
    "logging_steps": 10,
    "eval_strategy": "steps",
    "eval_steps": 50,
    "save_strategy": "steps",
    "save_steps": 50,
    "save_total_limit": 2,
    "gradient_checkpointing": True,
    "report_to": "none",
    "seed": 42
}

# Verify essential configuration options
essential = ["assistant_only_loss", "max_length"]

missing = [key for key in essential if key not in supported]

if missing:
    raise RuntimeError(
        f"Your TRL version lacks required options: {missing}. "
        "Update TRL before continuing."
    )

# Filter optional parameters unsupported by this version
filtered_config = {
    key: value
    for key, value in config.items()
    if key in supported
}

print("Unsupported optional parameters:",
      set(config) - set(filtered_config))

training_args = SFTConfig(**filtered_config)

print("SFT configuration created successfully!")


Unsupported optional parameters: set()
SFT configuration created successfully!


/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


In [38]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer
)

print("SFTTrainer initialized successfully!")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))


Tokenizing train dataset:   0%|          | 0/33273 [00:00<?, ? examples/s]

RuntimeError: You're using `assistant_only_loss=True`, but at least one example has no assistant tokens. This usually means the tokenizer's chat template doesn't generate assistant masks — it may be missing the `{% generation %}` keyword. Please check the template and ensure it's correctly configured to support assistant masking.

In [ ]:

sample = trainer.train_dataset[0]

print("Available columns:", sample.keys())

if "labels" in sample:
    labels = sample["labels"]

    active_labels = [
        token for token in labels
        if token != -100
    ]

    print("Trainable tokens:", len(active_labels))

    assert len(active_labels) > 0, (
        "No trainable assistant tokens found."
    )

    print("Assistant target:")
    print(tokenizer.decode(active_labels[:200]))

else:
    print(
        "Labels are not stored in the dataset. "
        "They may be created by the trainer's data collator."
    )
